# Fake News Detection Using Machine Learning & Natural Language Processing

## Academic Research & Experimental Benchmark Notebook

**Objective**: This notebook conducts an end-to-end comparative study on text classification algorithms for detecting deceptive versus credible news articles using Natural Language Processing (NLP) feature extraction and Scikit-learn statistical classifiers.

### Pipeline Stages:
1. **Exploratory Data Analysis (EDA)**: Class balance, lexical lengths, and corpus statistics.
2. **NLP Preprocessing**: Text normalization, URL removal, entity prefix cleaning, and stopword filtering.
3. **Feature Engineering**: Sublinear TF-IDF n-gram vectorization fitted strictly on training splits to prevent data leakage.
4. **Classifier Training**: Comparative evaluation of Logistic Regression, Multinomial Naive Bayes, Linear Support Vector Machines (Calibrated), and Random Forest.
5. **Performance Evaluation**: Accuracy, Precision, Recall, F1-Score, ROC-AUC, and Confusion Matrix visualization.
6. **Academic Limitation Analysis**: Discussion on lexical heuristics vs. factual truth grounding.

## 1. Import Libraries & Configure Environment

In [ ]:
import os
import re
import string
import json
import joblib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve, precision_recall_curve
)

# Ensure NLTK stopwords available
try:
    nltk.download('stopwords', quiet=True)
    STOPWORDS = set(stopwords.words('english'))
except Exception:
    STOPWORDS = set(['the', 'a', 'an', 'in', 'on', 'of', 'for', 'to', 'and', 'is', 'it'])

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Libraries successfully imported!")

## 2. Load & Prepare Datasets (`Fake.csv` & `True.csv`)

In [ ]:
data_dir = Path("../data")
fake_df = pd.read_csv(data_dir / "Fake.csv")
true_df = pd.read_csv(data_dir / "True.csv")

fake_df['label'] = 0
fake_df['label_name'] = 'FAKE'

true_df['label'] = 1
true_df['label_name'] = 'REAL'

# Combine datasets
df = pd.concat([fake_df, true_df], ignore_index=True)
df['combined_text'] = df['title'].fillna('') + ' ' + df['text'].fillna('')

# Remove duplicates and missing text
df = df.drop_duplicates(subset=['combined_text']).reset_index(drop=True)

print(f"Total unique articles: {len(df):,}")
print(f"Class Distribution:\n{df['label_name'].value_counts()}\n")
df.head()

## 3. Exploratory Data Analysis: Lexical & Word Count Distributions

In [ ]:
df['word_count'] = df['combined_text'].apply(lambda x: len(x.split()))
df['char_count'] = df['combined_text'].apply(len)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=df, x='word_count', hue='label_name', kde=True, ax=axes[0], palette=['#ef4444', '#10b981'], bins=30)
axes[0].set_title("Word Count Distribution by Class", fontsize=13, weight='bold')
axes[0].set_xlabel("Number of Words")
axes[0].set_ylabel("Article Frequency")

sns.boxplot(data=df, x='label_name', y='word_count', ax=axes[1], palette=['#ef4444', '#10b981'])
axes[1].set_title("Article Length Comparison (Box Plot)", fontsize=13, weight='bold')
axes[1].set_ylabel("Word Count")
axes[1].set_xlabel("Class Label")

plt.tight_layout()
plt.show()

## 4. NLP Text Preprocessing Pipeline

We apply standard text cleaning while ensuring no data leakage occurs:

In [ ]:
def preprocess_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'https?:\/\/\S+|www\.\S+', ' ', text)
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'^[a-z\s]+(?:\(reuters\)|\(ap\))\s*-\s*', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    tokens = text.split()
    filtered = [t for t in tokens if len(t) > 2 and t not in STOPWORDS]
    return ' '.join(filtered)

df['clean_text'] = df['combined_text'].apply(preprocess_text)
df = df[df['clean_text'].str.strip() != ''].reset_index(drop=True)

print(f"Cleaned Corpus Size: {len(df):,} samples")
print("\nSample Cleaned Text:", df['clean_text'].iloc[0][:150], "...")

## 5. Train / Test Split & Feature Extraction (TF-IDF)

**Strict No-Leakage Policy**: The `TfidfVectorizer` is instantiated and fitted exclusively on `X_train`, followed by out-of-sample transformation on `X_test`.

In [ ]:
X = df['clean_text'].values
y = df['label'].values

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=10000,
    min_df=2,
    max_df=0.85,
    sublinear_tf=True
)

X_train_vec = vectorizer.fit_transform(X_train_raw)
X_test_vec = vectorizer.transform(X_test_raw)

print(f"Training feature matrix shape: {X_train_vec.shape}")
print(f"Testing feature matrix shape:  {X_test_vec.shape}")

## 6. Model Training & Evaluation Benchmark

We train and benchmark 4 algorithms: Logistic Regression, Multinomial Naive Bayes, Calibrated Linear SVM, and Random Forest.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    "Multinomial Naive Bayes": MultinomialNB(alpha=0.1),
    "Linear SVM (Calibrated)": CalibratedClassifierCV(LinearSVC(C=1.0, max_iter=2000, random_state=42), cv=3),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=25, random_state=42, n_jobs=-1)
}

metrics_records = []
fitted_clfs = {}
proba_dict = {}

for name, clf in models.items():
    clf.fit(X_train_vec, y_train)
    y_pred = clf.predict(X_test_vec)
    
    if hasattr(clf, 'predict_proba'):
        y_proba = clf.predict_proba(X_test_vec)[:, 1]
    elif hasattr(clf, 'decision_function'):
        d = clf.decision_function(X_test_vec)
        y_proba = 1 / (1 + np.exp(-d))
    else:
        y_proba = y_pred
        
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_proba)
    
    fitted_clfs[name] = clf
    proba_dict[name] = (y_proba, roc_auc)
    
    metrics_records.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1,
        "ROC-AUC": roc_auc
    })

results_df = pd.DataFrame(metrics_records)
results_df

## 7. Comparative Visualizations: Confusion Matrices & ROC Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
axes = axes.flatten()

for idx, (name, clf) in enumerate(fitted_clfs.items()):
    y_pred = clf.predict(X_test_vec)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], cbar=False,
                xticklabels=['Pred FAKE', 'Pred REAL'], yticklabels=['True FAKE', 'True REAL'])
    axes[idx].set_title(name, fontsize=12, weight='bold')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 6))
for name, (y_proba, auc_val) in proba_dict.items():
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc_val:.4f})", linewidth=2.0)

plt.plot([0, 1], [0, 1], 'k--', alpha=0.6, label='Random Baseline')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves Comparison', fontsize=14, weight='bold')
plt.legend(loc='lower right')
plt.show()

## 8. Academic Discussion & Limitations

### Key Findings:
1. **Lexical Representation**: TF-IDF capture salient stylistic signals (e.g. dramatic adverbs, urgent calls to action in fabricated articles vs. standardized agency attributions in verified reporting).
2. **Classifier Comparison**: Logistic Regression and Calibrated Linear SVM offer high computational efficiency and robust generalization without overfitting high-dimensional n-gram spaces.
3. **Crucial Scientific Limitation**: This model detects statistical lexical patterns correlated with the labeled training dataset. **It does NOT verify real-world factual claims**. A carefully written false article using formal journalistic phrasing may be classified as 'REAL', while an authentic article written with passionate or informal prose could trigger 'FAKE' lexical markers. Automated NLP classification should be treated as an assistive diagnostic tool, not an absolute arbiter of truth.